# 04 - BERT Fine-Tuning

Fine-tunes BERT once on the frozen train/validation splits: tokenizes text, applies the
configured long-document strategy, detects meaningful class imbalance and applies
class-weighted loss (computed from training data only) when the configured threshold is met,
and selects the best checkpoint by validation macro F1.

**Long-document strategy decision (docs/BLUEPRINT.md Section 6):** both `first_512` and
`beginning_middle_end` are implemented. This notebook trains one model per strategy on the
same train/validation split and keeps whichever wins on **validation** macro F1 -- test-set
performance is never used for this choice. The winning strategy is written back to
`configs/base.yaml: long_document_strategy.default`, and only the winning model is saved as
the versioned artifact under `artifacts/models/`.

`test.csv` is not touched anywhere in this notebook -- that only happens once, in
`05_bert_evaluation.ipynb`.

All logic lives in `src/newstart_ai/models/bert/`; this notebook only calls it, trains,
compares, and saves.

### Load the frozen split and the BERT-related tools

**Purpose:** Load the train/validation split saved by notebook 03 (never the test split --
that stays untouched until notebook 05), plus everything needed to build, train, and save a
BERT classifier.

**Why this step is necessary:** This notebook must never construct its own split or peek at
`test.csv` -- `load_split()` reads the exact files `03_reproducible_splitting.ipynb` wrote to
disk, guaranteeing every notebook trains and evaluates on identical data. Printing
`settings.bert.base_model` here is a visible reminder that the checkpoint name comes from
`configs/bert.yaml`, not from a hard-coded string anywhere in this notebook or in
`src/newstart_ai/models/bert/`.

**Inputs:** `data/splits/{train,validation,test}.csv` and `split_manifest.json` (only
`train_df`/`val_df`/`manifest` are actually used below), plus `configs/bert.yaml` and
`configs/base.yaml`.

**Output:** `train_df`, `val_df` (loaded but `test_df` intentionally unused), `manifest`,
and `settings`.

**How to interpret the result:** The printed row counts should match notebook 03's saved
split exactly (482 train / 121 validation). `test_df` is loaded by `load_split()` for
convenience but this notebook never reads from it -- that's a deliberate discipline, not an
oversight.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

from newstart_ai.config import load_settings
from newstart_ai.data import load_split
from newstart_ai.models.bert import (
    BERTClassifier,
    build_long_document_strategy,
    new_artifact_id,
    save_artifact,
    BertArtifactMetadata,
)

settings = load_settings()
# load_split() reads the exact CSVs notebook 03 saved -- this notebook never recomputes or
# re-splits the data, and it never reads from test_df.
train_df, val_df, test_df, manifest = load_split(settings)
print(f"train: {len(train_df)}  validation: {len(val_df)}  (test set not loaded here)")
print(f"base_model (from configs/bert.yaml): {settings.bert.base_model}")

train: 482  validation: 121  (test set not loaded here)
base_model (from configs/bert.yaml): bert-base-uncased


## Train one model per long-document strategy

Both runs use identical data, hyperparameters, and base checkpoint -- only the long-document
strategy differs. Training progress (per-epoch train loss, validation loss, validation
accuracy, validation macro F1) prints below for each run.

### Train one full BERT model per long-document strategy

**Purpose:** Fine-tune `bert-base-uncased` twice -- once configured to use the
`first_512` long-document strategy, once using `beginning_middle_end` -- keeping every other
setting (data, hyperparameters, checkpoint) identical between the two runs, so the only
variable being compared is how each strategy handles documents longer than BERT's 512-token
limit.

**Why this step is necessary:** Notebook 02 showed that many documents are far longer than
BERT can read in one pass. Rather than assuming one truncation strategy is best, the project
design requires implementing and comparing both, then picking a winner using validation
performance only -- never by looking at the test set. Running both here, back to back, is
what makes that comparison possible.

**What happens inside `classifier.fit(train_df, val_df)`, and why:**
- **Class weighting:** before training starts, `fit()` computes how many training documents
  belong to each agency. Because IRS has far fewer documents than DMV or USCIS
  (an imbalance ratio of roughly 12x, first measured in notebook 02), the loss function is
  automatically weighted so that mistakes on the rare IRS class count for more than mistakes
  on a common class like DMV. This weighting is calculated *only* from `train_df` -- the
  validation and test sets are never used to influence what the model is trained to
  prioritize.
- **Long-document handling:** each document is converted into one or more fixed-size token
  windows by the selected strategy. `first_512` simply keeps a document's first 512 tokens.
  `beginning_middle_end` instead samples up to three windows -- the start, a middle section,
  and the end -- so a long document isn't represented solely by its opening paragraph. Every
  window inherits its parent document's label for training.
- **Epoch loop:** the model trains for `configs/bert.yaml: max_epochs` epochs. After each
  epoch, it's evaluated on `val_df` (never `test_df`), and the epoch with the best validation
  macro F1 is the one that's kept -- this is standard early-checkpoint-selection practice to
  avoid picking an overfit late epoch.

**Inputs:** `train_df`, `val_df`, and the two long-document strategy configurations.

**Output:** `results`, a dictionary keyed by strategy name, each holding the trained
`classifier`, its per-epoch `history` (train loss, validation loss, validation accuracy,
validation macro F1), and its single best validation macro F1 score.

**How to interpret the result:** Watch the per-epoch validation macro F1 climb (or
plateau) across epochs for each strategy -- that trend is what "best checkpoint selection"
is reacting to. The final printed number for each strategy is what the next cell compares
head-to-head; a tie or near-tie between the two strategies (as happened in this run) is
itself a meaningful result, not a failure of the comparison.

In [2]:
# Train one full model per candidate long-document strategy, changing nothing else, so the
# validation-macro-F1 comparison below isolates the effect of that one design choice.
strategy_names = ["first_512", "beginning_middle_end"]
results = {}

for strategy_name in strategy_names:
    print(f"=== Training with long-document strategy: {strategy_name} ===")
    # Builds the chunking strategy from configs/base.yaml -- swapping strategies here never
    # requires touching BERTClassifier or any other source code.
    strategy = build_long_document_strategy(settings, override=strategy_name)
    classifier = BERTClassifier(settings, long_document_strategy=strategy)

    # Inside fit(): class weights are computed from train_df's label counts only (never
    # validation/test), long documents are chunked per the selected strategy, and the
    # checkpoint with the best validation macro F1 across all epochs is kept automatically.
    training_result = classifier.fit(train_df, val_df)

    results[strategy_name] = {
        "classifier": classifier,
        "history": training_result["history"],
        "best_validation_macro_f1": training_result["best_validation_macro_f1"],
    }
    print(
        f"Best validation macro F1 for {strategy_name}: "
        f"{training_result['best_validation_macro_f1']:.4f}"
    )
    print()

=== Training with long-document strategy: first_512 ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1  train_loss=0.9923  val_loss=0.3282  val_acc=0.9587  val_macro_f1=0.9360


epoch 2  train_loss=0.2276  val_loss=0.0654  val_acc=1.0000  val_macro_f1=1.0000


...

epoch 2  train_loss=0.1032  val_loss=0.0422  val_acc=1.0000  val_macro_f1=1.0000


epoch 3  train_loss=0.0552  val_loss=0.0684  val_acc=0.9917  val_macro_f1=0.9941


epoch 4  train_loss=0.0198  val_loss=0.0211  val_acc=1.0000  val_macro_f1=1.0000


epoch 5  train_loss=0.0096  val_loss=0.0214  val_acc=1.0000  val_macro_f1=1.0000


epoch 6  train_loss=0.0607  val_loss=0.0321  val_acc=1.0000  val_macro_f1=1.0000
Best validation macro F1 for beginning_middle_end: 1.0000



## Compare strategies and select the winner

Selection uses validation macro F1 only -- test.csv has not been loaded in this notebook.

### Compare the two strategies side by side

**Purpose:** Lay the two strategies' best validation macro F1 scores next to each other in
one small table, sorted from best to worst.

**Why this step is necessary:** This is the concrete comparison step the project design
calls for -- turning two separate training runs' results into a single, explicit ranking
that the next cell can act on programmatically, rather than a human eyeballing printed
numbers scattered across the notebook.

**Inputs:** `results`, populated by the training loop above.

**Output:** `comparison`, a small DataFrame with one row per strategy, sorted best-first.

**How to interpret the result:** The top row (index 0) is the winning strategy. If both
scores are identical (as they were in this run, both reaching 1.0), the strategy listed
first in `strategy_names` (`first_512`) wins the tie by virtue of pandas' stable sort --
a deterministic, documented tie-break rather than an arbitrary one.

In [3]:
import pandas as pd

comparison = pd.DataFrame(
    {
        "strategy": strategy_names,
        "best_validation_macro_f1": [results[s]["best_validation_macro_f1"] for s in strategy_names],
    }
).sort_values("best_validation_macro_f1", ascending=False, ignore_index=True)
comparison

,strategy,best_validation_macro_f1
0,first_512,1.0
1,beginning_middle_end,1.0


### Select the winning strategy and its trained model

**Purpose:** Pull out the name of the best-performing strategy and the already-trained
classifier that goes with it.

**Why this step is necessary:** Everything from this point forward in the notebook --
persisting the decision to configuration, and saving the model artifact -- operates on the
winner only. The losing strategy's trained model is simply discarded; it was only ever
needed for this comparison.

**Inputs:** `comparison` (the sorted table above) and `results`.

**Output:** `winning_strategy` (a string) and `winning_classifier` (a trained
`BERTClassifier` instance).

**How to interpret the result:** The printed strategy name is the one that will be written
into configuration and used for every later notebook (05 evaluation, and eventually the demo
app) -- it is the project's single, resolved answer to "which long-document strategy should
BERT use?".

In [4]:
winning_strategy = comparison.iloc[0]["strategy"]
winning_classifier = results[winning_strategy]["classifier"]
print(f"Winning long-document strategy: {winning_strategy}")

Winning long-document strategy: first_512


## Persist the winning strategy in configuration

Updates `configs/base.yaml: long_document_strategy.default` so every later notebook (and the
demo app) uses the same decision without needing to repeat this comparison.

### Record the decision in configuration

**Purpose:** Write the winning strategy's name into `configs/base.yaml` under
`long_document_strategy.default`, replacing whatever placeholder value was there before.

**Why this step is necessary:** A decision made only inside this notebook's memory would be
invisible to every other notebook, the future API, and anyone reading the config files
later. Writing it back to the shared YAML config -- the same file every notebook loads
settings from -- makes the decision durable and visible without requiring any code changes
elsewhere.

**Inputs:** `winning_strategy` and the existing `configs/base.yaml` file on disk.

**Output:** An updated `configs/base.yaml` file; nothing in memory changes.

**How to interpret the result:** After this cell runs, any notebook that calls
`load_settings()` again will see the resolved strategy as the new default -- this is exactly
the same pattern used later in notebook 08 to record the demo's default routing method.

In [5]:
import yaml

base_yaml_path = settings.project_root / "configs" / "base.yaml"
with open(base_yaml_path, "r", encoding="utf-8") as f:
    raw_config = yaml.safe_load(f)

# Overwrite just the one field this notebook is responsible for deciding -- everything else
# in base.yaml is left exactly as it was.
raw_config["long_document_strategy"]["default"] = winning_strategy

with open(base_yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(raw_config, f, sort_keys=False)

print(f"configs/base.yaml long_document_strategy.default -> {winning_strategy}")

configs/base.yaml long_document_strategy.default -> first_512


## Save the versioned artifact

Only the winning model is saved. Metadata records the base model, label order, dataset
fingerprint, split manifest reference, long-document strategy, training configuration, and
validation metrics. `test_metrics` stays empty until `05_bert_evaluation.ipynb` runs -- and
the artifact is marked `ready` only once training has completed successfully, matching the
"only READY artifacts are usable" rule from the original design.

### Save the winning model as a versioned artifact

**Purpose:** Persist the winning model's weights, tokenizer, and a rich metadata record
(base model, label order, dataset fingerprint, training configuration, and validation
results) to `artifacts/models/<artifact_id>/`, marked `status="ready"`.

**Why this step is necessary:** Notebook 05 (and later, the demo app) must load *this exact*
trained model rather than retraining it -- artifacts are how a trained model becomes a
reusable, inspectable object instead of something that only exists for the lifetime of this
notebook's Python process. Recording the dataset fingerprint and training configuration
alongside the weights is what makes the artifact's provenance traceable later: anyone
looking at it can see exactly which strategy, which hyperparameters, and which version of
the dataset produced it. Marking it `"ready"` only after training and comparison have fully
finished means a partially-trained or losing-strategy model can never be mistaken for the
project's chosen model.

**Inputs:** `winning_classifier` (model + tokenizer), plus everything needed to describe how
it was produced (`settings`, `manifest`, `results`, `comparison`).

**Output:** A new directory under `artifacts/models/` containing the model checkpoint,
tokenizer files, and a `metadata.json`; the printed `artifact_id` is this model's permanent,
immutable identifier.

**How to interpret the result:** The printed `artifact_id` is what notebook 05 will look up
via `latest_ready_artifact_id()` -- there's no need to copy it by hand between notebooks.

In [6]:
from datetime import datetime, timezone

# A new, immutable ID for this trained model -- used as the on-disk folder name and the
# only trusted way to reference this specific artifact later (never the display name).
artifact_id = new_artifact_id()
metadata = BertArtifactMetadata(
    artifact_id=artifact_id,
    display_name="bert-mvp",
    base_model=settings.bert.base_model,
    label_order=settings.base.labels,
    dataset_fingerprint=manifest.dataset_fingerprint,
    long_document_strategy=winning_strategy,
    training_config={
        "max_epochs": settings.bert.max_epochs,
        "batch_size": settings.bert.batch_size,
        "learning_rate": settings.bert.learning_rate,
        "weighted_loss_threshold": settings.bert.imbalance.weighted_loss_threshold,
        "class_weights_applied": winning_classifier.class_weights is not None,
    },
    validation_metrics={
        "best_validation_macro_f1": results[winning_strategy]["best_validation_macro_f1"],
        "history": results[winning_strategy]["history"],
        "strategy_comparison": comparison.to_dict(orient="records"),
    },
    # "ready" means: trained, compared, and safe for 05_bert_evaluation (or later, the demo
    # app) to load. An artifact that failed or lost the strategy comparison never reaches
    # this status.
    status="ready",
    ready_at=datetime.now(timezone.utc).isoformat(),
)

artifact_path = save_artifact(winning_classifier.model, winning_classifier.tokenizer, metadata, settings)
print(f"Saved BERT artifact {artifact_id} to {artifact_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved BERT artifact 3628681550d7433b94407f684946bb2f to D:\USD\Projects\a590\newstart-ai\newstart_ai_benchmark\artifacts\models\3628681550d7433b94407f684946bb2f


## Summary for the next notebook

- Winning long-document strategy and BERT artifact are saved and marked READY.
- `05_bert_evaluation.ipynb` loads this exact artifact and evaluates it once on `test.csv`.
- IRS's small test slice (see `03_reproducible_splitting.ipynb`) means the IRS validation
  macro F1 above should also be read with the same small-sample caveat.